In [1]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image, ImageDraw
import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
DATA_DIR = Path("D:/ROGII")
CSV_PATH = DATA_DIR / "train_curves_b01.csv"
TIFF_DIR = DATA_DIR
DEFAULT_ID = "0018ca82"
DEFAULT_SEQUENCE = 1
DEFAULT_WINDOW_HEIGHT = 1400
X_PADDING = 35
Y_PADDING = 100
MASK_LINE_WIDTH = 2
OVERLAY_LINE_WIDTH = 2

In [3]:
df = pd.read_csv(CSV_PATH, dtype={"id": str})
df["id"] = df["id"].str.strip()
assert {"id", "curves"}.issubset(df.columns), \
    "В CSV должны присутствовать колонки id и curves"
print("Количество наблюдений:", len(df))
print("Размер CSV:", df.shape)

Количество наблюдений: 500
Размер CSV: (500, 3)


In [4]:
def load_observation_curves(observation_id):
    rows = df[df["id"] == str(observation_id).strip()]
    if len(rows) == 0:
        raise ValueError(
            f"Observation ID {observation_id!r} отсутствует в CSV"
        )
    if len(rows) > 1:
        raise ValueError(
            f"Найдено несколько записей для ID {observation_id}"
        )
    raw = rows.iloc[0]["curves"]
    if isinstance(raw, str):
        curves = json.loads(raw)
    else:
        curves = raw
    return curves

def load_tiff(observation_id):
    path = TIFF_DIR / f"{observation_id}.tif"
    if not path.exists():
        raise FileNotFoundError(f"Не найден TIFF: {path}")
    with Image.open(path) as img:
        img.seek(0)
        return img.copy()

def parse_sequence(sequence):
    segments = []
    current = []
    for point in sequence:
        if point is None:
            if current:
                segments.append(np.asarray(current, dtype=float))
            current = []
            continue
        x, y = point[:2]
        if x is None or y is None:
            if current:
                segments.append(np.asarray(current, dtype=float))
            current = []
            continue
        x, y = float(x), float(y)
        if not np.isfinite(x) or not np.isfinite(y):
            if current:
                segments.append(np.asarray(current, dtype=float))
            current = []
            continue
        current.append([x, y])
    if current:
        segments.append(np.asarray(current, dtype=float))
    if not segments:
        raise ValueError("В выбранной последовательности нет точек")
    return segments

def get_sequence_bounds(segments):
    points = np.vstack(segments)
    xmin = points[:, 0].min()
    xmax = points[:, 0].max()
    ymin = points[:, 1].min()
    ymax = points[:, 1].max()
    return xmin, xmax, ymin, ymax


def to_local_coordinates(segments, x0, y0):
    return [
        np.rint(
            segment - np.array([x0, y0])
        ).astype(int)
        for segment in segments
    ]

def create_mask(size, segments):
    width, height = size
    mask = Image.new("L", (width, height), 0)
    draw = ImageDraw.Draw(mask)
    for points in segments:
        points = [
            tuple(map(int, point))
            for point in points
        ]
        if len(points) == 1:
            draw.point(points[0], fill=255)
        elif len(points) >= 2:
            draw.line(
                points,
                fill=255,
                width=MASK_LINE_WIDTH
            )
    return mask

def create_overlay(image, segments):
    result = image.convert("RGB").copy()
    draw = ImageDraw.Draw(result)
    for points in segments:
        points = [
            tuple(map(int, point))
            for point in points
        ]
        if len(points) == 1:
            draw.point(points[0], fill=(255, 0, 0))
        elif len(points) >= 2:
            draw.line(
                points,
                fill=(255, 0, 0),
                width=OVERLAY_LINE_WIDTH
            )
    return result

In [5]:
id_widget = widgets.Text(
    value=DEFAULT_ID,
    description="Observation ID:",
    placeholder="0018ca82",
    continuous_update=False,
    layout=widgets.Layout(width="280px")
)
sequence_widget = widgets.BoundedIntText(
    value=DEFAULT_SEQUENCE,
    min=1,
    max=1,
    description="Sequence №:",
    layout=widgets.Layout(width="190px")
)
window_widget = widgets.BoundedIntText(
    value=DEFAULT_WINDOW_HEIGHT,
    min=100,
    max=20000,
    step=100,
    description="Window height:",
    layout=widgets.Layout(width="230px")
)
load_button = widgets.Button(
    description="Load observation",
    button_style="primary",
    icon="refresh"
)
scroll_widget = widgets.IntSlider(
    value=0,
    min=0,
    max=0,
    step=1,
    description="Y",
    orientation="vertical",
    continuous_update=False,
    disabled=True,
    readout=True,
    layout=widgets.Layout(
        height="520px",
        width="90px"
    )
)
status_output = widgets.Output()
top_output = widgets.Output(
    layout=widgets.Layout(width="100%")
)
bottom_output = widgets.Output(
    layout=widgets.Layout(width="92%")
)
STATE = {
    "id": None,
    "image": None,
    "curves": None,
    "segments": None,
    "bounds": None,
    "sequence_index": None,
    "x0": None,
    "x1": None,
    "ymin": None,
    "ymax": None,
    "window_height": None,
    "busy": False
}
controls = widgets.HBox(
    [
        id_widget,
        sequence_widget,
        window_widget,
        load_button
    ],
    layout=widgets.Layout(
        display="flex",
        flex_flow="row wrap",
        align_items="center"
    )
)
bottom_layout = widgets.HBox(
    [
        bottom_output,
        scroll_widget
    ],
    layout=widgets.Layout(
        width="100%",
        align_items="center"
    )
)
interface = widgets.VBox(
    [
        controls,
        status_output,
        top_output,
        bottom_layout
    ],
    layout=widgets.Layout(width="100%")
)
display(interface)

In [6]:
def render_top():
    image = STATE["image"]
    xmin, xmax, ymin, ymax = STATE["bounds"]
    observation_id = STATE["id"]
    sequence_index = STATE["sequence_index"]
    H = image.height
    rotated = image.transpose(
        Image.Transpose.ROTATE_270
    )
    arr = np.asarray(rotated)
    fig, ax = plt.subplots(
        figsize=(17, 3),
        dpi=120
    )
    if arr.ndim == 2:
        ax.imshow(
            arr,
            cmap="gray",
            origin="upper",
            aspect="auto",
            interpolation="nearest"
        )
    else:
        ax.imshow(
            arr,
            origin="upper",
            aspect="auto",
            interpolation="nearest"
        )
    rect_x = H - 1 - ymax
    rect_y = xmin
    rect_width = max(1, ymax - ymin + 1)
    rect_height = max(1, xmax - xmin + 1)
    rectangle = Rectangle(
        (rect_x - 0.5, rect_y - 0.5),
        rect_width,
        rect_height,
        linewidth=2,
        edgecolor="red",
        facecolor="none"
    )
    ax.add_patch(rectangle)
    ax.set_title(
        f"Observation: {observation_id} | "
        f"Sequence: {sequence_index} | "
        f"Full TIFF rotated clockwise"
    )
    ax.axis("off")
    plt.tight_layout()
    with top_output:
        clear_output(wait=True)
        display(fig)
    plt.close(fig)

In [7]:
def render_bottom():
    image = STATE["image"]
    x0 = STATE["x0"]
    x1 = STATE["x1"]
    window_height = STATE["window_height"]
    y0 = int(scroll_widget.value)
    y1 = min(image.height, y0 + window_height)
    crop_box = (x0, y0, x1 + 1, y1)
    tiff_crop = image.crop(crop_box)
    local_segments = to_local_coordinates(
        STATE["segments"],
        x0,
        y0
    )
    mask = create_mask(
        tiff_crop.size,
        local_segments
    )
    overlay = create_overlay(
        tiff_crop,
        local_segments
    )
    tiff_array = np.asarray(tiff_crop)
    mask_array = np.asarray(mask)
    overlay_array = np.asarray(overlay)
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(19, 9),
        dpi=120
    )
    panels = [
        (axes[0], tiff_array, "1. Original TIFF"),
        (axes[1], mask_array, "2. Selected sequence mask"),
        (axes[2], overlay_array, "3. TIFF + CSV annotation")
    ]
    crop_height = y1 - y0
    crop_width = x1 - x0 + 1
    for ax, arr, title in panels:
        if arr.ndim == 2:
            ax.imshow(
                arr,
                cmap="gray",
                origin="upper",
                aspect="auto",
                interpolation="nearest"
            )
        else:
            ax.imshow(
                arr,
                origin="upper",
                aspect="auto",
                interpolation="nearest"
            )
        ax.set_title(title)
        ax.set_xlim(-0.5, crop_width - 0.5)
        ax.set_ylim(crop_height - 0.5, -0.5)
        ax.set_xlabel("X (local pixels)")
        ax.grid(False)
    axes[0].set_ylabel("Y (local pixels)")
    fig.suptitle(
        f"ID: {STATE['id']} | "
        f"Sequence: {STATE['sequence_index']} | "
        f"Y: {y0} — {y1}",
        fontsize=13
    )
    plt.tight_layout(
        rect=[0, 0, 1, 0.95]
    )
    with bottom_output:
        clear_output(wait=True)
        display(fig)
    plt.close(fig)
    with status_output:
        clear_output(wait=True)
        print(
            f"Observation: {STATE['id']} | "
            f"Sequence: {STATE['sequence_index']} / "
            f"{len(STATE['curves'])}"
        )
        print(
            f"X: {x0} — {x1} | "
            f"Y: {y0} — {y1} | "
            f"Crop: {crop_width} × {crop_height} px"
        )

In [8]:
def configure_scroll():
    image = STATE["image"]
    image_height = image.height
    xmin, xmax, ymin, ymax = STATE["bounds"]
    x0 = max(
        0,
        int(math.floor(xmin)) - X_PADDING
    )
    x1 = min(
        image.width - 1,
        int(math.ceil(xmax)) + X_PADDING
    )
    window_height = min(
        int(window_widget.value),
        image_height
    )
    max_top = image_height - window_height
    scroll_min = max(
        0,
        int(math.floor(ymin)) - Y_PADDING
    )
    scroll_max = min(
        max_top,
        int(math.ceil(ymax)) + Y_PADDING - window_height
    )
    scroll_min = min(scroll_min, max_top)
    scroll_max = max(scroll_min, scroll_max)
    initial_y = int(math.floor(ymin)) - Y_PADDING
    initial_y = int(
        np.clip(
            initial_y,
            scroll_min,
            scroll_max
        )
    )
    STATE["x0"] = x0
    STATE["x1"] = x1
    STATE["window_height"] = window_height
    STATE["ymin"] = scroll_min
    STATE["ymax"] = scroll_max
    STATE["busy"] = True
    scroll_widget.min = 0
    scroll_widget.max = max_top
    scroll_widget.value = initial_y
    scroll_widget.min = scroll_min
    scroll_widget.max = scroll_max
    scroll_widget.value = initial_y
    scroll_widget.disabled = (
        scroll_min == scroll_max
    )
    STATE["busy"] = False

def activate_sequence(sequence_number):
    if STATE["curves"] is None:
        return
    if not (1 <= sequence_number <= len(STATE["curves"])):
        raise ValueError(
            f"Sequence должна быть в диапазоне "
            f"1..{len(STATE['curves'])}"
        )
    sequence = STATE["curves"][sequence_number - 1]
    segments = parse_sequence(sequence)
    bounds = get_sequence_bounds(segments)
    STATE["segments"] = segments
    STATE["bounds"] = bounds
    STATE["sequence_index"] = sequence_number
    configure_scroll()
    render_top()
    render_bottom()

In [ ]:
def load_observation(_=None):
    try:
        STATE["busy"] = True
        observation_id = id_widget.value.strip()
        if not observation_id:
            raise ValueError("Введите Observation ID")
        curves = load_observation_curves(observation_id)
        image = load_tiff(observation_id)
        if len(curves) == 0:
            raise ValueError("У наблюдения нет последовательностей")
        sequence_widget.max = len(curves)
        sequence_number = int(sequence_widget.value)
        if sequence_number > len(curves):
            sequence_number = 1
        sequence_widget.value = sequence_number
        STATE["id"] = observation_id
        STATE["image"] = image
        STATE["curves"] = curves
        STATE["busy"] = False
        activate_sequence(sequence_number)
    except Exception as error:
        STATE["busy"] = False
        with status_output:
            clear_output(wait=True)
            print(
                f"Error: {type(error).__name__}: {error}"
            )

def on_sequence_change(change):
    if STATE["busy"]:
        return
    if STATE["image"] is None:
        return
    try:
        activate_sequence(
            int(change["new"])
        )
    except Exception as error:
        with status_output:
            clear_output(wait=True)
            print(
                f"Sequence error: {type(error).__name__}: {error}"
            )

def on_scroll_change(change):
    if STATE["busy"]:
        return
    if STATE["image"] is None:
        return
    if change["name"] == "value":
        render_bottom()

def on_window_height_change(change):
    if STATE["busy"]:
        return
    if STATE["image"] is None:
        return
    try:
        configure_scroll()
        render_bottom()
    except Exception as error:
        with status_output:
            clear_output(wait=True)
            print(
                f"Window error: {type(error).__name__}: {error}"
            )

load_button.on_click(load_observation)
sequence_widget.observe(
    on_sequence_change,
    names="value"
)
scroll_widget.observe(
    on_scroll_change,
    names="value"
)
window_widget.observe(
    on_window_height_change,
    names="value"
)
load_observation()